In [0]:
%sql
-- Run this first in every notebook session
USE CATALOG mini_project_grp6;

In [0]:
%sql
-- Bronze: raw ingestion layer
CREATE SCHEMA IF NOT EXISTS mini_project_grp6.bronze
  COMMENT 'Raw ingestion layer — unmodified source data with metadata columns';

-- Silver: cleaned and computed layer
CREATE SCHEMA IF NOT EXISTS mini_project_grp6.silver
  COMMENT 'Cleaned and computed layer — sessions, engagement, dropout risk';

-- Gold: business-ready aggregates
CREATE SCHEMA IF NOT EXISTS mini_project_grp6.gold
  COMMENT 'BI-ready layer — course performance, instructor dashboard, completion funnel';

-- Audit: data quality audit records
CREATE SCHEMA IF NOT EXISTS mini_project_grp6.audit
  COMMENT 'DQ audit table — failed rows, flags, freshness SLA tracking';

In [0]:
%sql
-- LMS events: Auto Loader will watch this path
CREATE VOLUME IF NOT EXISTS mini_project_grp6.bronze.lms_events_raw
  COMMENT 'Landing zone for lms_events NDJSON files — read by Auto Loader';

-- Video watch events: batch Parquet
CREATE VOLUME IF NOT EXISTS mini_project_grp6.bronze.video_watch_raw
  COMMENT 'Landing zone for video_watch_events.parquet daily export';

-- Quiz attempts: incremental CSV
CREATE VOLUME IF NOT EXISTS mini_project_grp6.bronze.quiz_attempts_raw
  COMMENT 'Landing zone for quiz_attempts.csv daily batch files';

-- Discussion posts: NDJSON
CREATE VOLUME IF NOT EXISTS mini_project_grp6.bronze.discussion_posts_raw
  COMMENT 'Landing zone for discussion_posts.json NDJSON files';

-- Course catalog: XLSX monthly upload
CREATE VOLUME IF NOT EXISTS mini_project_grp6.bronze.course_catalog_raw
  COMMENT 'Landing zone for course_catalog.xlsx and .csv monthly admin file';

-- Student enrollments: daily delta CSV
CREATE VOLUME IF NOT EXISTS mini_project_grp6.bronze.enrollments_raw
  COMMENT 'Landing zone for student_enrollments.csv daily delta files';

-- Checkpoints: Auto Loader stream state (lms + video)
CREATE VOLUME IF NOT EXISTS mini_project_grp6.bronze.checkpoints
  COMMENT 'Auto Loader checkpoint and schema inference storage';

In [0]:
%sql
-- Check schemas
SHOW SCHEMAS IN mini_project_grp6;

-- Check volumes in bronze
SHOW VOLUMES IN mini_project_grp6.bronze;

-- Confirm volume paths are accessible (run in Python cell)
-- dbutils.fs.ls("/Volumes/mini_project_grp6/bronze/lms_events_raw/")
-- dbutils.fs.ls("/Volumes/mini_project_grp6/bronze/checkpoints/")

In [0]:
%sql
-- %sql cell: set default catalog and schema at session start
USE CATALOG mini_project_grp6;
USE SCHEMA bronze;  -- change to silver / gold as needed per notebook

In [0]:
# -- %python cell: set Spark defaults (so you can omit catalog prefix in spark.table())
# spark.conf.set("spark.sql.catalog.mini_project_grp6", "databricks.unityCatalog")
# spark.catalog.setCurrentCatalog("mini_project_grp6")
# spark.catalog.setCurrentDatabase("bronze")  # change per notebook

# -- Volume path constants (use these in every notebook)
LMS_RAW_PATH        = "/Volumes/mini_project_grp6/bronze/lms_events_raw/"
VIDEO_RAW_PATH      = "/Volumes/mini_project_grp6/bronze/video_watch_raw/"
QUIZ_RAW_PATH       = "/Volumes/mini_project_grp6/bronze/quiz_attempts_raw/"
DISCUSSION_RAW_PATH = "/Volumes/mini_project_grp6/bronze/discussion_posts_raw/"
CATALOG_RAW_PATH    = "/Volumes/mini_project_grp6/bronze/course_catalog_raw/"
ENROLLMENTS_RAW_PATH= "/Volumes/mini_project_grp6/bronze/enrollments_raw/"
CHECKPOINT_BASE     = "/Volumes/mini_project_grp6/bronze/checkpoints/"